---

## 🧩 什么时候用 `@dynamic_prompt`

| 场景 | 读什么 | 例子 |
|---|---|---|
| 🌍 语言/地区 | `request.runtime.context` | 用用户的语言回复 |
| 👤 用户角色 | `request.runtime.context` | 管理员 vs 客户的不同指令 |
| 📊 对话阶段 | `request.messages` | 引导 → 解决问题 → 收尾 |
| 🕐 按时间 | `datetime.now()` | "现在是非工作时间，建议查 FAQ" |
| 🔧 感知工具 | `request.tools` | "你现在能用的工具是：X、Y、Z" |
| 🗂️ 感知 state | `request.runtime.state` | "用户已经给过 X 了，别重复问" |

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

💡 **动手试试**：`@dynamic_prompt` 可以和其他 middleware 自由组合——比如系统提示用 `@dynamic_prompt`，消息管理用 `SummarizationMiddleware`，两者不冲突。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## ⚙️ 用 `@dynamic_prompt` 做动态提示词

**一句话定位**：系统提示不该是写死的字符串，而是能根据运行时信息实时生成的函数。

</div>

### 🤔 静态提示词的问题

用 `system_prompt="You are a helpful assistant."` 创建 agent，这句提示是**固定**的——每个用户、每一轮、每种上下文都拿到一样的指令。但真实应用里提示词经常要跟着这些东西变：

- 🌍 用户的语言/地区
- 👤 用户角色（管理员 / 客户 / 访客）
- 🕐 时间段 / 是否工作时间
- 📊 agent 已经了解到的信息（对话状态）
- 🔧 这次会话里有哪些工具可用

### ✅ 解法：`@dynamic_prompt`

`@dynamic_prompt` 是一种 config 风格的 middleware，根据当前的 `ModelRequest` **在运行时生成系统提示**，替代静态的 `system_prompt=` 参数。

```python
@dynamic_prompt
def my_prompt(request: ModelRequest) -> str:
    # 可以读 context、state、messages、tools——任何需要的东西
    role = request.runtime.context.user_role
    return f"You are a {role}-level assistant."
```

### `ModelRequest` 能读到什么

| 字段 | 内容 |
|---|---|
| `request.messages` | 当前对话历史 |
| `request.runtime.context` | 只读上下文（来自 invoke 时的 `context=`） |
| `request.runtime.state` | 可变的 agent state |
| `request.tools` | agent 当前能用的工具 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**：不要同时设置 `system_prompt=` 和 `@dynamic_prompt`——动态的那个会覆盖静态的，容易以为两者会合并，结果调了半天发现静态提示根本没生效。

</div>

---

## 📝 例 1：按语言生成提示词

最简单的用例——根据用户上下文切换回复语言。

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dataclass
class LanguageContext:
    user_language: str = "English"


@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Dia dhuit! Tá mé go maith, go raibh maith agat. Conas atá tú?


In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

¡Hola! Estoy bien, gracias. ¿Y tú? ¿En qué puedo ayudarte hoy?


In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

Bonjour ! Je vais très bien, merci. Et toi, comment vas-tu ?


---

## 📝 例 2：按角色生成提示词

不同用户拿到不同指令：管理员看到技术细节，客户拿到友好的客服话术。

In [ ]:
@dataclass
class UserContext:
    user_role: str = "customer"   # "admin", "customer", or "guest"
    user_name: str = "Friend"


@dynamic_prompt
def role_based_prompt(request: ModelRequest) -> str:
    """Tailor the system prompt based on user role."""
    ctx = request.runtime.context
    role = ctx.user_role
    name = ctx.user_name

    if role == "admin":
        return (
            f"You are a technical assistant for admin {name}. "
            "Be precise, include error codes, stack traces, and config details. "
            "Assume the user is an expert."
        )
    elif role == "customer":
        return (
            f"You are a friendly support agent helping {name}. "
            "Use simple language, avoid jargon, and be empathetic. "
            "Always offer to escalate if the issue is complex."
        )
    else:  # guest
        return (
            "You are a helpful assistant. The user is not logged in. "
            "Provide general information only. Do not access account data."
        )


agent_role = create_agent(
    model="gpt-5-nano",
    context_schema=UserContext,
    middleware=[role_based_prompt],
)

In [ ]:
# Admin gets technical, detailed response
response = agent_role.invoke(
    {"messages": [HumanMessage(content="The server is returning 502 errors")]},
    context=UserContext(user_role="admin", user_name="Alice"),
)
print("🔧 Admin response:")
print(response["messages"][-1].content)

In [ ]:
# Customer gets friendly, simple response
response = agent_role.invoke(
    {"messages": [HumanMessage(content="The website isn't working")]},
    context=UserContext(user_role="customer", user_name="Bob"),
)
print("😊 Customer response:")
print(response["messages"][-1].content)

---

## 📝 例 3：感知对话状态的提示词

提示词随着**agent 在对话里已经了解到的信息**变化。这里靠 `request.messages` 检查对话历史长度。

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：`@dynamic_prompt` 把"系统提示该写什么"从一个写死的字符串，变成一个可以读 context / state / messages / tools 的函数——这是让同一个 agent 应对不同用户、不同阶段、不同权限的核心手段之一。

</div>

In [ ]:
from langchain.messages import AIMessage


@dynamic_prompt
def conversation_phase_prompt(request: ModelRequest) -> str:
    """Change the prompt based on how far along the conversation is."""
    msg_count = len(request.messages)

    if msg_count <= 2:
        return (
            "You are a friendly onboarding assistant. "
            "Ask the user's name and what they need help with. "
            "Be warm and welcoming."
        )
    elif msg_count <= 6:
        return (
            "You are now in problem-solving mode. "
            "The user has introduced themselves. "
            "Focus on understanding their issue. Ask clarifying questions."
        )
    else:
        return (
            "You are in resolution mode. "
            "You've gathered enough context. "
            "Provide a clear, actionable solution. Be concise."
        )


agent_phased = create_agent(
    model="gpt-5-nano",
    middleware=[conversation_phase_prompt],
)

In [ ]:
# Phase 1: Onboarding (≤2 messages)
response = agent_phased.invoke(
    {"messages": [HumanMessage(content="Hi there")]}
)
print("Phase 1 (onboarding):")
print(response["messages"][-1].content)

In [ ]:
# Phase 2: Problem-solving (3-6 messages)
response = agent_phased.invoke(
    {"messages": [
        HumanMessage(content="Hi there"),
        AIMessage(content="Welcome! What's your name?"),
        HumanMessage(content="I'm Alice. My app keeps crashing."),
    ]}
)
print("Phase 2 (problem-solving):")
print(response["messages"][-1].content)

In [ ]:
# Phase 3: Resolution (7+ messages)
response = agent_phased.invoke(
    {"messages": [
        HumanMessage(content="Hi there"),
        AIMessage(content="Welcome! What's your name?"),
        HumanMessage(content="I'm Alice. My app keeps crashing."),
        AIMessage(content="What device and OS are you using?"),
        HumanMessage(content="iPhone 15, iOS 18."),
        AIMessage(content="When does it crash? On launch or during use?"),
        HumanMessage(content="It crashes when I open the camera."),
    ]}
)
print("Phase 3 (resolution):")
print(response["messages"][-1].content)